# WHO-HealthPropaPhenKG Evaluation

In [1]:
%load_ext autoreload
%autoreload 2

## Introduction

In [2]:
#!pip install unicode

### Standard

In [3]:
import numpy as np
import pandas as pd

## Paths

Download the following files:  
url = "https://covid19.who.int/WHO-COVID-19-global-data.csv"  
url_iso = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/UID_ISO_FIPS_LookUp_Table.csv"

In [4]:
path_who_data = "../data/groudtruth/WHO-COVID-19-global-data.csv"
path_isodata = "../data/groudtruth/UID_ISO_FIPS_LookUp_Table.csv"
path_coviddata = "../data/groudtruth/covidToWorld.csv"

In [5]:
mpox_url = "https://raw.githubusercontent.com/owid/monkeypox/main/owid-monkeypox-data.csv"
path_mpoxdata = "../data/groudtruth/mpoxToWorld.csv"

In [6]:
path_to_netwoork_gazetteer = '../data/gazetteers/world_gazetteer_en.csv'

In [7]:
import unidecode

In [8]:
def preprocessCountryNames(df, columnName):
    df[columnName] = df[columnName].apply(lambda x: str(x).replace('"','').lower())
    df[columnName] = df[columnName].apply(lambda x: x.split(' (')[0])
    df[columnName] = df[columnName].apply(lambda x: x.replace('the ',''))
    df.loc[df[columnName] == 'viet nam',columnName] = "vietnam" 
    df.loc[df[columnName] == 'republic of korea',columnName] = "south korea" 
    df.loc[df[columnName] == 'united republic of tanzania',columnName] = "tanzania" 
    df.loc[df[columnName] == 'russian federation',columnName] = "russia" 
    df.loc[df[columnName] == "côte d'ivoire",columnName] = "ivory coast"
    df.loc[df[columnName] == "congo",columnName] = "republic of congo"
    df.loc[df[columnName] == "brunei darussalam",columnName] = "brunei"
    df.loc[df[columnName] == "türkiye",columnName] = "turkey"
    df.loc[df[columnName] == "syrian arab republic",columnName] = "syria"
    df.loc[df[columnName] == "united kingdom of great britain and northern ireland",columnName] = "united kingdom"
    df.loc[df[columnName] == "united states of america",columnName] = "united states"
    df.loc[df[columnName] == "lao people's democratic republic",columnName] = "laos"
    df.loc[df[columnName] == "timor-leste",columnName] = "east timor"
    df.loc[df[columnName] == "micronesia",columnName] = "federated states of micronesia"
    df.loc[df[columnName] == "republic of moldova",columnName] = "moldova"
    df[columnName] = df[columnName].apply(lambda x: unidecode.unidecode(x))
    return df

### Reading Gazetteer

In [9]:
df_network = pd.read_csv(path_to_netwoork_gazetteer)

In [10]:
df_network

,Unnamed: 0,ID,Name
0,0,wkg:10,"""Mamassita"""
1,1,wkg:10,"""Mamacita"""
2,2,wkg:1000709658,"""Boulzazen"""
3,3,wkg:1000709658,"""Boulzazen"""
4,4,wkg:1000709660,"""Tizi El Oued"""
...,...,...,...
1692478,1692478,wkg:985505906,"""Mehuín"""
1692479,1692479,wkg:992034877,"""San Juana"""
1692480,1692480,wkg:995446460,"""Cucutilla"""
1692481,1692481,wkg:999504607,"""Prusia"""


In [11]:
df_network = df_network.drop(columns=['Unnamed: 0'])

In [12]:
df_network = preprocessCountryNames(df_network, "Name")

In [13]:
df_network

,ID,Name
0,wkg:10,mamassita
1,wkg:10,mamacita
2,wkg:1000709658,boulzazen
3,wkg:1000709658,boulzazen
4,wkg:1000709660,tizi el oued
...,...,...
1692478,wkg:985505906,mehuin
1692479,wkg:992034877,san juana
1692480,wkg:995446460,cucutilla
1692481,wkg:999504607,prusia


## Reading WHO COVID data

In [14]:
data = pd.read_csv(path_who_data)

In [15]:
data = data[data["Cumulative_cases"] > 0]

In [16]:
data = preprocessCountryNames(data, "Country")

In [17]:
data = data.drop(['Country_code', 'WHO_region', 'New_cases', 'New_deaths',
       'Cumulative_deaths'],axis=1)

In [18]:
data=data.rename(columns = {'Date_reported':'date'})

In [19]:
data

,date,Country,Cumulative_cases
8,2020-03-01,afghanistan,1
9,2020-03-08,afghanistan,1
10,2020-03-15,afghanistan,7
11,2020-03-22,afghanistan,24
12,2020-03-29,afghanistan,91
...,...,...,...
52315,2024-02-04,zimbabwe,266319
52316,2024-02-11,zimbabwe,266328
52317,2024-02-18,zimbabwe,266350
52318,2024-02-25,zimbabwe,266355


In [20]:
data["Country"].unique().shape

(238,)

In [21]:
maindata = pd.merge(data,df_network,left_on="Country",right_on="Name")
maindata = maindata.drop(['Name', 'Country'],axis=1)

In [22]:
maindata['CUI:START_ID']= maindata.shape[0]*['C5203670']
maindata[':TYPE']= maindata.shape[0]*["isReported"]
maindata = maindata.rename(columns={"ID": "ID:END_ID"})

In [23]:
maindata = maindata.drop_duplicates(keep=False)

In [24]:
maindata

,date,Cumulative_cases,ID:END_ID,CUI:START_ID,:TYPE
0,2020-03-01,1,wkg:26847706,C5203670,isReported
1,2020-03-08,1,wkg:26847706,C5203670,isReported
2,2020-03-15,7,wkg:26847706,C5203670,isReported
3,2020-03-22,24,wkg:26847706,C5203670,isReported
4,2020-03-29,91,wkg:26847706,C5203670,isReported
...,...,...,...,...,...
139086,2024-02-25,349304,wkg:5338938963,C5203670,isReported
139087,2024-02-25,349304,wkg:6436501440,C5203670,isReported
139090,2024-03-03,349304,wkg:4860232524,C5203670,isReported
139091,2024-03-03,349304,wkg:5338938963,C5203670,isReported


In [25]:
maindata.to_csv(path_coviddata)

## Reading WHO Mpox data

In [26]:
mpoxdata = pd.read_csv(
    mpox_url,
    sep=',',
    encoding='utf-8',
)

In [27]:
mpoxdata.head()

,location,iso_code,date,total_cases,total_deaths,new_cases,new_deaths,new_cases_smoothed,new_deaths_smoothed,new_cases_per_million,total_cases_per_million,new_cases_smoothed_per_million,new_deaths_per_million,total_deaths_per_million,new_deaths_smoothed_per_million
0,Africa,OWID_AFR,2022-05-01,27.0,2.0,0.0,0.0,0.29,0.0,0.0,0.019,0.0,0.0,0.0014,0.0
1,Africa,OWID_AFR,2022-05-02,27.0,2.0,0.0,0.0,0.29,0.0,0.0,0.019,0.0,0.0,0.0014,0.0
2,Africa,OWID_AFR,2022-05-03,27.0,2.0,0.0,0.0,0.29,0.0,0.0,0.019,0.0,0.0,0.0014,0.0
3,Africa,OWID_AFR,2022-05-04,27.0,2.0,0.0,0.0,0.29,0.0,0.0,0.019,0.0,0.0,0.0014,0.0
4,Africa,OWID_AFR,2022-05-05,27.0,2.0,0.0,0.0,0.29,0.0,0.0,0.019,0.0,0.0,0.0014,0.0


In [28]:
def monkeypoxDataEnhancement(data):
    # Drop useless columns
    maindata = data.drop(['iso_code','total_deaths',
      'new_cases', 'new_deaths', 'new_cases_smoothed', 'new_deaths_smoothed',
      'new_cases_per_million', 'total_cases_per_million',
      'new_cases_smoothed_per_million', 'new_deaths_per_million',
      'total_deaths_per_million', 'new_deaths_smoothed_per_million'],axis=1)
    #  Columns
    all_dates = maindata["date"].unique().tolist()
    all_dates.sort()
    date_list = []
    total_cases = []
    iso_code = []
    dict_countries = dict(zip(maindata["location"].unique().tolist(), [0]*len(maindata["location"])))
    # Go through every date
    for date in all_dates:
      # Get slice of matrix
      slice = data[data["date"]==date]
      # Get lines of slice
      for index, row in slice.iterrows():
        dict_countries[row['location']] = row['total_cases']
      # Add values to lists
      for key in dict_countries.keys():
        date_list.append(date)
        iso_code.append(key)
        total_cases.append(dict_countries[key])
    # Create pandas dataframe
    d = {'location':iso_code,'date':date_list, 'total_cases':total_cases}
    df = pd.DataFrame(data=d)
    # Get all isocode
    return df

In [29]:
mainmpoxdata = monkeypoxDataEnhancement(mpoxdata)
mainmpoxdata = mainmpoxdata[mainmpoxdata['total_cases']>0]
mainmpoxdata["location"] = mainmpoxdata["location"].apply(lambda x: x.replace('"','').lower())
mainmpoxdata.loc[mainmpoxdata["location"] == 'united states',"location"] = "united states of america"

In [30]:
mainmpoxdata = preprocessCountryNames(mainmpoxdata,'location')

In [31]:
mainmpoxdata["location"].unique()

array(['africa', 'cameroon', 'canada', 'central african republic',
       'republic of congo', 'ghana', 'nigeria', 'north america',
       'united states', 'world', 'europe', 'united kingdom', 'ireland',
       'mexico', 'portugal', 'sweden', 'belgium', 'italy', 'spain',
       'australia', 'democratic republic of congo', 'finland', 'france',
       'germany', 'oceania', 'asia', 'israel', 'switzerland',
       'netherlands', 'austria', 'denmark', 'czechia', 'slovenia',
       'united arab emirates', 'malta', 'argentina', 'colombia',
       'hungary', 'norway', 'south america', 'gibraltar', 'morocco',
       'latvia', 'brazil', 'greece', 'iceland', 'poland', 'chile',
       'romania', 'venezuela', 'georgia', 'luxembourg', 'serbia',
       'lebanon', 'singapore', 'south korea', 'bulgaria', 'benin',
       'china', 'croatia', 'bahamas', 'estonia', 'peru', 'puerto rico',
       'turkey', 'south africa', 'dominican republic', 'ecuador',
       'jamaica', 'panama', 'slovakia', 'new zealand',

In [32]:
mainmpoxdataplus = pd.merge(mainmpoxdata,df_network,left_on="location",right_on="Name")
mainmpoxdataplus = mainmpoxdataplus.drop(['Name', 'location'],axis=1)

In [33]:
mainmpoxdataplus['CUI:START_ID']= mainmpoxdataplus.shape[0]*['C0276180']
mainmpoxdataplus[':TYPE']= mainmpoxdataplus.shape[0]*["isReported"]
mainmpoxdataplus = mainmpoxdataplus.rename(columns={"ID": "ID:END_ID"})

In [34]:
mainmpoxdataplus = mainmpoxdataplus.drop_duplicates(keep=False)

In [35]:
mainmpoxdataplus

,date,total_cases,ID:END_ID,CUI:START_ID,:TYPE
0,2022-05-01,27.0,wkg:5039605589,C0276180,isReported
1,2022-05-01,27.0,wkg:5829877885,C0276180,isReported
4,2022-05-02,27.0,wkg:5039605589,C0276180,isReported
5,2022-05-02,27.0,wkg:5829877885,C0276180,isReported
8,2022-05-03,27.0,wkg:5039605589,C0276180,isReported
...,...,...,...,...,...
277594,2024-05-17,13.0,wkg:432424952,C0276180,isReported
277595,2024-05-18,13.0,wkg:432424952,C0276180,isReported
277596,2024-05-19,13.0,wkg:432424952,C0276180,isReported
277597,2024-05-20,13.0,wkg:432424952,C0276180,isReported


In [36]:
mainmpoxdataplus.to_csv(path_mpoxdata)